# ColorCheck selector validation

Train only a small quality selector on new scene-disjoint target/reference pairs. The v3 LUT and existing statistics matcher are frozen. GPU is used for candidate images and color metrics; logistic regression is small. This run uses training and validation photos only; the final split remains sealed.

In [ ]:
from pathlib import Path
import torch, sklearn
assert torch.cuda.is_available(), 'Enable a Kaggle GPU'
for directory in ('tools', 'tools/ml', 'tools/ml/reference_matcher', 'tools/ml/tst100k'):
    path = Path('/kaggle/working') / directory
    path.mkdir(parents=True, exist_ok=True)
    (path / '__init__.py').touch()
print({'torch': torch.__version__, 'sklearn': sklearn.__version__, 'gpu': torch.cuda.get_device_name(0)})


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/model.py
"""Small reference-conditioned model that predicts a browser-applicable 3D LUT."""

from __future__ import annotations

import torch
from torch import nn


class ConvBlock(nn.Sequential):
    """Downsample an image while learning increasingly abstract visual features."""

    def __init__(self, incoming: int, outgoing: int) -> None:
        super().__init__(
            # Depthwise convolution examines spatial neighborhoods separately in
            # each channel; the 1x1 convolution then mixes channels efficiently.
            nn.Conv2d(incoming, incoming, 3, stride=2, padding=1, groups=incoming),
            nn.Conv2d(incoming, outgoing, 1),
            nn.GroupNorm(min(8, outgoing), outgoing),
            nn.SiLU(),
        )


class ImageEncoder(nn.Module):
    """Summarize a photograph as a compact 96-number feature vector."""

    def __init__(self, feature_size: int = 96) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, stride=2, padding=2),
            nn.SiLU(),
            ConvBlock(16, 24),
            ConvBlock(24, 40),
            ConvBlock(40, 64),
            ConvBlock(64, feature_size),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        # Adaptive pooling reduces any supported image size to one value per
        # feature channel. ``flatten(1)`` changes BxCx1x1 into BxC.
        return self.features(image).flatten(1)


class ExpertStyleClassifier(nn.Module):
    """Training-only head that asks encoder features to retain editing style.

    FiveK's A-E labels are supervision for representation learning, not product
    choices exposed to users. The head is saved for reproducibility but is never
    called by ``ReferenceLutModel.forward`` and therefore never enters ONNX.
    """

    def __init__(self, feature_size: int = 96, expert_count: int = 5) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(feature_size, 64),
            nn.SiLU(),
            nn.Linear(64, expert_count),
        )

    def forward(self, reference_features: torch.Tensor) -> torch.Tensor:
        return self.layers(reference_features)


def identity_lut(size: int) -> torch.Tensor:
    """Create a 3D lookup table that returns every RGB color unchanged."""
    values = torch.linspace(0, 1, size)
    blue, green, red = torch.meshgrid(values, values, values, indexing="ij")
    return torch.stack((red, green, blue))


def initial_lut_bases(size: int, basis_count: int) -> torch.Tensor:
    """Create distinct, paired photographic anchors with a near-neutral mean.

    Identically initialized LUTs make mixture coefficients almost irrelevant and
    starve the reference predictor of useful gradients. These mild warm/cool,
    contrast, saturation, and gamma pairs break that symmetry without imposing
    an extreme look. Every value remains trainable after initialization.
    """
    identity = identity_lut(size)
    luminance = (
        identity * identity.new_tensor([0.2126, 0.7152, 0.0722])[:, None, None, None]
    ).sum(0, keepdim=True)
    anchors = [
        identity * identity.new_tensor([1.06, 1.0, 0.94])[:, None, None, None],
        identity * identity.new_tensor([0.94, 1.0, 1.06])[:, None, None, None],
        (identity - 0.5) * 1.08 + 0.5,
        (identity - 0.5) * 0.92 + 0.5,
        luminance + (identity - luminance) * 1.12,
        luminance + (identity - luminance) * 0.88,
        identity.clamp_min(0).pow(0.92),
        identity.clamp_min(0).pow(1.08),
    ]
    selected = [anchors[index % len(anchors)] for index in range(basis_count)]
    return torch.stack(selected).clamp(0, 1)


class ReferenceLutModel(nn.Module):
    """Predict a safe blend of learned LUTs from target and reference features.

    A 3D lookup table maps an input RGB triplet to a new RGB triplet. The model
    learns several reusable LUT bases, then predicts non-negative coefficients
    that sum to one. This convex blend is bounded and compact enough for browser
    inference, unlike a network that generates every full-resolution pixel.
    """

    def __init__(self, basis_count: int = 8, lut_size: int = 33) -> None:
        super().__init__()
        self.basis_count = basis_count
        self.lut_size = lut_size
        self.histogram_bins = 8
        self.encoder = ImageEncoder()
        self.predictor = nn.Sequential(
            # Five learned feature interactions, channel statistics, and two
            # four-channel soft histograms describe both scene and treatment.
            nn.Linear(96 * 5 + 12 + 2 * 4 * self.histogram_bins, 160),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(160, basis_count),
        )
        self.basis = nn.Parameter(initial_lut_bases(lut_size, basis_count))

    @staticmethod
    def statistics(image: torch.Tensor) -> torch.Tensor:
        """Describe each RGB channel with its mean and standard deviation."""
        mean = image.mean(dim=(2, 3))
        std = image.std(dim=(2, 3), correction=0)
        return torch.cat((mean, std), dim=1)

    def color_distribution(self, image: torch.Tensor) -> torch.Tensor:
        """Summarize RGB and luminance distributions with soft histogram bins.

        Means and standard deviations cannot distinguish, for example, an image
        concentrated in shadows and highlights from one concentrated in the
        midtones. Each pixel contributes linearly to its neighboring bins, so
        this descriptor retains more tonal information while remaining smooth,
        differentiable, and exportable through ordinary ONNX operations.
        """
        luminance = (
            image * image.new_tensor([0.2126, 0.7152, 0.0722])[None, :, None, None]
        ).sum(1, keepdim=True)
        channels = torch.cat((image, luminance), dim=1).unsqueeze(2)
        centers = torch.linspace(
            0,
            1,
            self.histogram_bins,
            device=image.device,
            dtype=image.dtype,
        ).view(1, 1, self.histogram_bins, 1, 1)
        membership = (1 - (channels - centers).abs() * (self.histogram_bins - 1)).relu()
        return membership.mean(dim=(3, 4)).flatten(1)

    def predict_coefficients(self, source: torch.Tensor, reference: torch.Tensor):
        """Decide how strongly each learned LUT basis should influence the result."""
        # One batched encoder call produces a smaller ONNX graph and reduces
        # browser runtime dispatch overhead compared with two duplicated calls.
        paired_features = self.encoder(torch.cat((source, reference), dim=0))
        source_features, reference_features = paired_features.chunk(2, dim=0)
        # The model sees each image, their feature difference, and simple color
        # statistics. Their combination answers both "what is present?" and
        # "how does the desired reference differ from this source?"
        features = torch.cat(
            (
                source_features,
                reference_features,
                reference_features - source_features,
                (reference_features - source_features).abs(),
                reference_features * source_features,
                self.statistics(source),
                self.statistics(reference),
                self.color_distribution(source),
                self.color_distribution(reference),
            ),
            dim=1,
        )
        scores = self.predictor(features)
        # This is a numerically stable softmax written as primitive operations.
        # ONNX Runtime 1.19's static quantizer cannot calibrate a Softmax node,
        # while ReduceMax/Exp/ReduceSum/Div preserve the same smooth convex
        # weights in PyTorch, native ONNX, and ONNX Runtime Web.
        shifted = scores - scores.amax(dim=1, keepdim=True)
        exponentials = shifted.exp()
        return exponentials / exponentials.sum(dim=1, keepdim=True)

    def forward(self, source: torch.Tensor, reference: torch.Tensor):
        coefficients = self.predict_coefficients(source, reference)
        bounded_basis = self.basis.clamp(0, 1)
        lut = torch.einsum("bk,kcxyz->bcxyz", coefficients, bounded_basis)
        return lut, coefficients


def apply_lut(image: torch.Tensor, lut: torch.Tensor) -> torch.Tensor:
    """Apply a 3D LUT with differentiable trilinear interpolation.

    An RGB color usually falls between LUT grid points. The eight surrounding
    colors are sampled and blended according to distance. Because PyTorch can
    differentiate through these blends, training can update both the LUT bases
    and the network that chooses their coefficients.
    """
    batch, channels, height, width = image.shape
    size = lut.shape[-1]
    scaled = image.clamp(0, 1) * (size - 1)
    lower = scaled.floor().long()
    upper = (lower + 1).clamp(max=size - 1)
    # Autocast may produce a BF16/FP16 LUT while the source image remains FP32.
    # ``lerp`` requires its interpolation weights to match the sampled LUT dtype.
    fraction = (scaled - lower).to(dtype=lut.dtype)
    red0, green0, blue0 = lower[:, 0], lower[:, 1], lower[:, 2]
    red1, green1, blue1 = upper[:, 0], upper[:, 1], upper[:, 2]
    red_weight, green_weight, blue_weight = (
        fraction[:, 0],
        fraction[:, 1],
        fraction[:, 2],
    )
    flattened = lut.reshape(batch, channels, -1)

    def sample(red, green, blue):
        # Flattening converts a 3D coordinate into one gather index. This avoids
        # a Python loop over pixels and keeps interpolation accelerator-friendly.
        index = (blue * size * size + green * size + red).reshape(batch, 1, -1)
        return flattened.gather(2, index.expand(-1, channels, -1)).reshape(
            batch, channels, height, width
        )

    c000 = sample(red0, green0, blue0)
    c100 = sample(red1, green0, blue0)
    c010 = sample(red0, green1, blue0)
    c110 = sample(red1, green1, blue0)
    c001 = sample(red0, green0, blue1)
    c101 = sample(red1, green0, blue1)
    c011 = sample(red0, green1, blue1)
    c111 = sample(red1, green1, blue1)
    rw, gw, bw = red_weight[:, None], green_weight[:, None], blue_weight[:, None]
    c00 = c000.lerp(c100, rw)
    c10 = c010.lerp(c110, rw)
    c01 = c001.lerp(c101, rw)
    c11 = c011.lerp(c111, rw)
    return c00.lerp(c10, gw).lerp(c01.lerp(c11, gw), bw)


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/losses.py
"""Training objectives for faithful color transfer with structural guardrails."""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn


def rgb_to_lab(image: torch.Tensor) -> torch.Tensor:
    """Convert display RGB into perceptual CIE Lab values.

    Equal RGB errors do not always look equally different to a person. Lab is
    used here because Euclidean distance in this space is a more meaningful
    color-error signal than raw channel distance.
    """
    # Undo the nonlinear sRGB transfer curve before converting color spaces.
    linear = torch.where(
        image <= 0.04045,
        image / 12.92,
        ((image + 0.055) / 1.055).pow(2.4),
    )
    matrix = image.new_tensor(
        [
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041],
        ]
    )
    xyz = torch.einsum("ij,bjhw->bihw", matrix, linear)
    xyz = xyz / image.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    delta = 6 / 29
    f = torch.where(
        xyz > delta**3, xyz.clamp_min(1e-8).pow(1 / 3), xyz / (3 * delta**2) + 4 / 29
    )
    lightness = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return torch.stack((lightness, a, b), dim=1)


def image_gradients(image: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Measure horizontal and vertical luminance edges in a batch of images."""
    luminance = (
        image * image.new_tensor([0.2126, 0.7152, 0.0722])[None, :, None, None]
    ).sum(1, keepdim=True)
    return luminance[:, :, :, 1:] - luminance[:, :, :, :-1], luminance[
        :, :, 1:, :
    ] - luminance[:, :, :-1, :]


def reference_ranking_loss(
    correct_output: torch.Tensor,
    wrong_output: torch.Tensor,
    expected: torch.Tensor,
    margin: float = 0.01,
) -> torch.Tensor:
    """Require the correct reference to beat a known-wrong style per image.

    Reconstruction alone can reward one broadly attractive enhancement. This
    pairwise hinge objective instead asks for the expected edit to be at least
    margin closer with its true reference. It is used only during training and
    adds no operation or model parameter to browser inference.
    """
    correct_error = (correct_output - expected).abs().mean(dim=(1, 2, 3))
    wrong_error = (wrong_output - expected).abs().mean(dim=(1, 2, 3))
    return (margin + correct_error - wrong_error).relu().mean()


def perceptual_reference_ranking_loss(
    correct_output: torch.Tensor,
    wrong_output: torch.Tensor,
    expected: torch.Tensor,
    margin: float = 0.25,
) -> torch.Tensor:
    """Rank references by perceptual Lab error instead of raw RGB distance.

    The strict deployment gate uses perceptual color difference. This companion
    objective gives training a signal in the same units: a correct reference
    should beat a known-wrong style by at least a small visible margin. The loss
    is training-only and therefore does not enlarge browser inference.
    """
    expected_lab = rgb_to_lab(expected)
    correct_delta = (rgb_to_lab(correct_output) - expected_lab).square().sum(1).sqrt()
    wrong_delta = (rgb_to_lab(wrong_output) - expected_lab).square().sum(1).sqrt()
    correct_error = correct_delta.mean(dim=(1, 2))
    wrong_error = wrong_delta.mean(dim=(1, 2))
    return (margin + correct_error - wrong_error).relu().mean()


def exact_lut_supervision_loss(
    predicted_lut: torch.Tensor,
    expected_lut: torch.Tensor,
    available: torch.Tensor,
) -> torch.Tensor:
    """Measure full-cube error only where an exact synthetic LUT is known."""
    per_sample = (predicted_lut.float() - expected_lut.float()).abs().flatten(1).mean(1)
    weights = available.to(device=per_sample.device, dtype=per_sample.dtype)
    return (per_sample * weights).sum() / weights.sum().clamp_min(1)


def same_style_consistency_loss(
    first_lut: torch.Tensor, second_lut: torch.Tensor
) -> torch.Tensor:
    """Make two references with the same treatment produce compatible LUTs.

    Smooth L1 tolerates small scene-dependent differences while penalizing a
    predictor that follows reference content instead of its editing treatment.
    """
    return F.smooth_l1_loss(first_lut.float(), second_lut.float(), beta=0.02)


def expert_style_classification_loss(
    logits: torch.Tensor, expert: torch.Tensor, is_real_edit: torch.Tensor
) -> torch.Tensor:
    """Train FiveK style recognition without assigning labels to synthetic looks."""
    selected = is_real_edit.to(device=logits.device, dtype=torch.bool)
    if not selected.any():
        # Preserve a differentiable zero so all-synthetic batches remain valid.
        return logits.sum() * 0
    return F.cross_entropy(logits[selected], expert.to(logits.device)[selected])


class ReferenceMatchLoss(nn.Module):
    """Combine visual accuracy with guardrails against destructive LUTs.

    No single number captures a good correction. Reconstruction and Lab error
    reward agreement with the expert edit; structure protects source edges;
    smoothness discourages color discontinuities; and monotonicity discourages
    inversions where a brighter input unexpectedly becomes darker.
    """

    def __init__(
        self,
        color_weight: float = 0.02,
        structure_weight: float = 2.0,
        smoothness_weight: float = 0.0005,
        monotonicity_weight: float = 0.002,
    ) -> None:
        super().__init__()
        self.color_weight = color_weight
        self.structure_weight = structure_weight
        self.smoothness_weight = smoothness_weight
        self.monotonicity_weight = monotonicity_weight

    def forward(self, output, expected, source, lut):
        # Charbonnier reconstruction behaves like L1 error while remaining
        # smooth around zero, which makes gradient-based optimization stable.
        reconstruction = torch.sqrt((output - expected).square() + 1e-6).mean()
        color = (
            (rgb_to_lab(output) - rgb_to_lab(expected))
            .square()
            .sum(1)
            .add(1e-6)
            .sqrt()
            .mean()
        )
        output_x, output_y = image_gradients(output)
        source_x, source_y = image_gradients(source)
        structure = F.l1_loss(output_x, source_x) + F.l1_loss(output_y, source_y)
        differences = [
            lut[:, :, 1:] - lut[:, :, :-1],
            lut[:, :, :, 1:] - lut[:, :, :, :-1],
            lut[:, :, :, :, 1:] - lut[:, :, :, :, :-1],
        ]
        smoothness = sum(diff.square().mean() for diff in differences)
        monotonicity = sum((-diff).relu().mean() for diff in differences)
        total = (
            reconstruction
            + self.color_weight * color
            + self.structure_weight * structure
            + self.smoothness_weight * smoothness
            + self.monotonicity_weight * monotonicity
        )
        return total, {
            "reconstruction": reconstruction.detach(),
            "delta_e76": color.detach(),
            "structure": structure.detach(),
            "lut_smoothness": smoothness.detach(),
            "lut_monotonicity": monotonicity.detach(),
        }


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/dataset.py
"""Load Adobe FiveK without leaking the same scene across data splits.

The training code supports a ZIP archive for local experiments and an extracted
directory for Kaggle. Both layouts contain ``raw`` originals and the five expert
edits in directories ``a`` through ``e``.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import zipfile
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF

# Adobe FiveK provides five independent professional edits for each photograph.
EXPERTS = ("a", "b", "c", "d", "e")
REQUIRED_DIRECTORIES = ("raw", *EXPERTS)


def apply_synthetic_style(image: torch.Tensor, seed: int) -> torch.Tensor:
    """Apply one deterministic, photograph-safe global editing recipe.

    The same seed gives two unrelated raw scenes the same treatment. Using one
    transformed scene as the reference and the other as the expected target
    creates exact supervision for arbitrary reference matching. This complements
    FiveK's real expert edits, whose intent is valuable but not parameterized.
    """
    generator = torch.Generator().manual_seed(seed)
    values = torch.rand(10, generator=generator, dtype=image.dtype)
    exposure = 2 ** float(values[0] * 2 - 1)
    contrast = float(0.72 + values[1] * 0.66)
    gamma = float(0.82 + values[2] * 0.38)
    saturation = float(0.65 + values[3] * 0.75)
    channel_gains = 0.86 + values[4:7, None, None] * 0.28
    shadow_lift = float(values[7] * 0.20 - 0.10)
    highlight_lift = float(values[8] * 0.20 - 0.10)
    black_lift = float(values[9] * 0.06 - 0.03)

    styled = image * channel_gains * exposure
    luminance = (
        styled * styled.new_tensor([0.2126, 0.7152, 0.0722])[:, None, None]
    ).sum(0, keepdim=True)
    styled = luminance + (styled - luminance) * saturation
    styled = (styled - 0.5) * contrast + 0.5
    lift = (
        shadow_lift * (1 - luminance.clamp(0, 1)).square()
        + highlight_lift * luminance.clamp(0, 1).square()
        + black_lift * (1 - luminance.clamp(0, 1)).pow(4)
    )
    styled = (styled + lift).clamp(0, 1).pow(gamma)
    return styled.clamp(0, 1)


def synthetic_style_lut(size: int, seed: int) -> torch.Tensor:
    """Materialize the exact 3D LUT implied by one synthetic editing recipe.

    Supervising the complete color cube teaches colors absent from a particular
    training crop and matches the artifact deployed in the browser. Axis order
    is channel, blue, green, red, the same contract used by apply_lut.
    """
    values = torch.linspace(0, 1, size)
    blue, green, red = torch.meshgrid(values, values, values, indexing="ij")
    identity = torch.stack((red, green, blue))
    styled = apply_synthetic_style(identity.reshape(3, size * size, size), seed)
    return styled.reshape(3, size, size, size)


@dataclass(frozen=True)
class SplitManifest:
    """Scene names assigned to training, validation, and final testing.

    Splitting by scene is essential: all six versions of one photograph must
    remain together, or evaluation would contain images already seen in training.
    """

    train: tuple[str, ...]
    validation: tuple[str, ...]
    test: tuple[str, ...]

    def save(self, path: Path) -> None:
        """Save the exact split so later runs evaluate the same photographs."""
        payload = {
            "version": 1,
            "method": "sha256(seed + scene filename), sorted by digest",
            "train": list(self.train),
            "validation": list(self.validation),
            "test": list(self.test),
        }
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2) + "\n")


def _contains_fivek_folders(path: Path) -> bool:
    """Return whether a directory directly contains raw and all expert folders."""
    return path.is_dir() and all(
        (path / name).is_dir() for name in REQUIRED_DIRECTORIES
    )


def find_dataset_root(path: Path) -> Path:
    """Find the FiveK folder inside a Kaggle dataset mount.

    Kaggle mounts datasets under a slug such as
    ``/kaggle/input/adobe-fivek``. This function also tolerates one or more
    wrapper directories so the notebook does not depend on an accidental path.
    """
    path = Path(path)
    if _contains_fivek_folders(path):
        return path

    candidates: list[Path] = []
    if path.is_dir():
        for raw_directory in path.rglob("raw"):
            if _contains_fivek_folders(raw_directory.parent):
                candidates.append(raw_directory.parent.resolve())

    unique = sorted(set(candidates))
    if len(unique) == 1:
        return unique[0]
    if not unique:
        raise ValueError(
            f"Could not find raw/a/b/c/d/e folders beneath dataset path: {path}"
        )
    raise ValueError(
        "More than one FiveK layout was found. Pass the exact directory with "
        f"--dataset-root. Candidates: {', '.join(map(str, unique))}"
    )


def inspect_dataset(dataset_path: Path) -> tuple[str, ...]:
    """Return scene filenames that have an original and all five expert edits."""
    dataset_path = Path(dataset_path)
    if dataset_path.is_file():
        if not zipfile.is_zipfile(dataset_path):
            raise ValueError(f"Dataset file is not a readable ZIP: {dataset_path}")
        with zipfile.ZipFile(dataset_path) as bundle:
            members = {
                name.replace("\\", "/")
                for name in bundle.namelist()
                if not name.endswith("/")
            }
        source_names = sorted(
            name.removeprefix("raw/") for name in members if name.startswith("raw/")
        )
        complete = [
            name
            for name in source_names
            if all(f"{expert}/{name}" in members for expert in EXPERTS)
        ]
    else:
        root = find_dataset_root(dataset_path)
        source_names = sorted(
            str(file.relative_to(root / "raw")).replace("\\", "/")
            for file in (root / "raw").rglob("*")
            if file.is_file()
        )
        complete = [
            name
            for name in source_names
            if all((root / expert / name).is_file() for expert in EXPERTS)
        ]

    missing_count = len(source_names) - len(complete)
    if missing_count:
        raise ValueError(
            f"Dataset has {missing_count} source scenes missing expert edits"
        )
    if not complete:
        raise ValueError("No complete raw/A-E scenes found in the dataset")
    return tuple(complete)


def inspect_archive(archive: Path) -> tuple[str, ...]:
    """Keep the original public function name for older local commands."""
    return inspect_dataset(archive)


def create_splits(
    scenes: tuple[str, ...],
    seed: int = 42,
    validation_size: int = 500,
    test_size: int = 500,
) -> SplitManifest:
    """Create deterministic, scene-level train/validation/test partitions."""
    if validation_size + test_size >= len(scenes):
        raise ValueError("Validation and test splits leave no training scenes")

    # Hash ranking makes the split independent of filesystem ordering while the
    # seed lets a future experiment deliberately choose a different partition.
    ranked = sorted(
        scenes,
        key=lambda name: hashlib.sha256(f"{seed}:{name}".encode()).digest(),
    )
    test = tuple(ranked[:test_size])
    validation = tuple(ranked[test_size : test_size + validation_size])
    train = tuple(ranked[test_size + validation_size :])
    return SplitManifest(train=train, validation=validation, test=test)


class FiveKReferenceDataset(Dataset):
    """Produce supervised examples for reference-conditioned color matching.

    Every sample contains:

    * ``source``: an unedited photograph that ColorCheck must transform;
    * ``reference``: a different scene edited by one particular expert; and
    * ``expected``: that expert's edit of the source photograph.

    Using a different reference scene prevents the network from copying scene
    content and instead asks it to infer the expert's visual treatment.
    """

    def __init__(
        self,
        dataset_path: Path,
        scenes: tuple[str, ...],
        image_size: int = 256,
        training: bool = False,
        seed: int = 42,
        synthetic_probability: float = 0.0,
        lut_size: int = 33,
        include_lut_supervision: bool = False,
        include_alternate_reference: bool = False,
    ) -> None:
        if len(scenes) < 2:
            raise ValueError("At least two scenes are required for reference pairing")
        self.dataset_path = Path(dataset_path)
        self.dataset_root = (
            None
            if self.dataset_path.is_file()
            else find_dataset_root(self.dataset_path)
        )
        self.scenes = scenes
        self.image_size = image_size
        self.training = training
        self.seed = seed
        if not 0 <= synthetic_probability <= 1:
            raise ValueError("Synthetic probability must be between zero and one")
        self.synthetic_probability = synthetic_probability
        self.lut_size = lut_size
        self.include_lut_supervision = include_lut_supervision
        self.include_alternate_reference = include_alternate_reference
        # ZIP handles cannot be shared safely by DataLoader worker processes.
        self._bundle: zipfile.ZipFile | None = None

    def __getstate__(self):
        """Remove an open ZIP handle before DataLoader copies this object."""
        state = self.__dict__.copy()
        state["_bundle"] = None
        return state

    def __len__(self) -> int:
        """Expose every scene once for each of the five expert styles."""
        return len(self.scenes) * len(EXPERTS)

    @property
    def bundle(self) -> zipfile.ZipFile:
        """Open the archive lazily inside the process that will read it."""
        if self._bundle is None:
            self._bundle = zipfile.ZipFile(self.dataset_path)
        return self._bundle

    def _prepare_image(self, image: Image.Image) -> torch.Tensor:
        """Convert one image into the consistent tensor format used by PyTorch."""
        image = image.convert("RGB")
        image = TF.resize(
            image,
            [self.image_size, self.image_size],
            interpolation=TF.InterpolationMode.BICUBIC,
            antialias=True,
        )
        # PIL uses height-width-channel integers in [0, 255]. ``to_tensor``
        # returns channel-height-width float values normalized to [0, 1].
        return TF.to_tensor(image)

    @lru_cache(maxsize=384)
    def _read(self, member: str) -> torch.Tensor:
        """Decode and cache a bounded working-size image inside each loader worker.

        FiveK reuses every raw scene across five expert samples and may reuse an
        expert image as another sample's reference. Caching the resized tensor
        avoids repeated JPEG decoding while keeping memory bounded. Returned
        tensors are never modified in place.
        """
        if self.dataset_root is None:
            with self.bundle.open(member) as stream:
                with Image.open(io.BytesIO(stream.read())) as image:
                    return self._prepare_image(image)
        with Image.open(self.dataset_root / member) as image:
            return self._prepare_image(image)

    def __getitem__(self, index: int):
        """Map a flat DataLoader index to a scene, expert, and reference scene."""
        scene_index, expert_index = divmod(index, len(EXPERTS))
        scene = self.scenes[scene_index]
        expert = EXPERTS[expert_index]

        # A stable hash selects a repeatable reference that can never equal the
        # source scene. Repeatability is especially important for validation.
        offset_seed = int.from_bytes(
            hashlib.sha256(f"{self.seed}:{scene}:{expert}".encode()).digest()[:8], "big"
        )
        reference_index = (
            scene_index + 1 + offset_seed % (len(self.scenes) - 1)
        ) % len(self.scenes)
        reference_scene = self.scenes[reference_index]

        # Training may request a second, same-style reference from another scene.
        # Agreement across two unrelated photographs teaches the model to retain
        # the edit treatment while discarding reference-scene content. Validation
        # and browser inference still require only one reference image.
        alternate_reference_scene = None
        if self.include_alternate_reference:
            alternate_seed = int.from_bytes(
                hashlib.sha256(
                    f"alternate:{self.seed}:{scene}:{expert}".encode()
                ).digest()[:8],
                "big",
            )
            alternate_index = (
                scene_index + 1 + alternate_seed % (len(self.scenes) - 1)
            ) % len(self.scenes)
            if len(self.scenes) > 2:
                while alternate_index in (scene_index, reference_index):
                    alternate_index = (alternate_index + 1) % len(self.scenes)
            alternate_reference_scene = self.scenes[alternate_index]

        source = self._read(f"raw/{scene}")
        # Deterministically mix exact synthetic supervision with real edits.
        # Five expert-indexed samples per scene provide five distinct seeds.
        style_seed = int.from_bytes(
            hashlib.sha256(f"style:{self.seed}:{scene}:{expert}".encode()).digest()[:8],
            "big",
        )
        synthetic = (style_seed % 10_000) / 10_000 < self.synthetic_probability
        if synthetic:
            reference = apply_synthetic_style(
                self._read(f"raw/{reference_scene}"), style_seed
            )
            expected = apply_synthetic_style(source, style_seed)
            if alternate_reference_scene is not None:
                alternate_reference = apply_synthetic_style(
                    self._read(f"raw/{alternate_reference_scene}"), style_seed
                )
        else:
            reference = self._read(f"{expert}/{reference_scene}")
            expected = self._read(f"{expert}/{scene}")
            if alternate_reference_scene is not None:
                alternate_reference = self._read(
                    f"{expert}/{alternate_reference_scene}"
                )

        # Flip source and expected together. This adds compositional variety
        # without changing the color relationship that the network must learn.
        if self.training and random.random() < 0.5:
            source = source.flip(-1)
            expected = expected.flip(-1)

        sample = {
            "source": source,
            "reference": reference,
            "expected": expected,
            "scene": scene,
            "reference_scene": reference_scene,
            "expert": expert_index,
            "supervision": "synthetic" if synthetic else "fivekExpert",
            # A style identifier supports the training-time wrong-reference
            # control. Synthetic recipes are unique; real edits share an expert.
            "style_id": f"synthetic:{style_seed}" if synthetic else f"expert:{expert}",
        }
        if self.include_lut_supervision:
            sample["expected_lut"] = (
                synthetic_style_lut(self.lut_size, style_seed)
                if synthetic
                else torch.zeros(3, self.lut_size, self.lut_size, self.lut_size)
            )
            sample["has_exact_lut"] = synthetic
        if alternate_reference_scene is not None:
            sample["alternate_reference"] = alternate_reference
            sample["alternate_reference_scene"] = alternate_reference_scene
        return sample


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/evaluation.py
"""Held-out evaluation for reference-conditioned photographic matching.

Training loss answers whether gradient descent can improve the model. This
module answers the harder product question: does the trained model improve an
unseen photograph, use the supplied reference, preserve structure, and remain
small and fast enough for an interactive browser editor?

Every metric is retained per photograph before aggregation. Confidence intervals
resample source scenes, not individual expert variants or pixels, because the five
edits of one FiveK photograph are related observations rather than five independent
photographs.
"""

from __future__ import annotations

import math
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F

from .dataset import EXPERTS
from .losses import image_gradients, rgb_to_lab

# These thresholds are declared in source before full training. They prevent a
# large evaluation set from promoting a statistically positive but visually tiny
# result. Confidence intervals must still exclude zero independently.
MINIMUM_PRACTICAL_EFFECTS = {
    "identityDeltaE2000": 0.25,
    "identitySsim": 0.002,
    "statisticsDeltaE2000": 0.10,
    "correctReferenceDeltaE2000": 0.05,
    "referenceSensitivityMae": 1 / 1024,
}


def _per_image_mean(values: torch.Tensor) -> torch.Tensor:
    """Average every dimension except the leading batch dimension."""
    return values.flatten(1).mean(1)


def _luminance(image: torch.Tensor) -> torch.Tensor:
    """Convert sRGB channels to a one-channel Rec. 709 luminance estimate."""
    weights = image.new_tensor([0.2126, 0.7152, 0.0722])[None, :, None, None]
    return (image * weights).sum(1, keepdim=True)


def ciede2000(lab_a: torch.Tensor, lab_b: torch.Tensor) -> torch.Tensor:
    """Return the CIEDE2000 color difference for corresponding Lab pixels.

    CIEDE2000 accounts for nonuniform sensitivity to lightness, chroma, and hue.
    It is used only as an evaluation metric: the smoother Delta E 76 objective
    remains preferable for backpropagation.
    """
    l1, a1, b1 = lab_a.unbind(1)
    l2, a2, b2 = lab_b.unbind(1)
    c1 = torch.sqrt(a1.square() + b1.square() + 1e-12)
    c2 = torch.sqrt(a2.square() + b2.square() + 1e-12)
    mean_c = (c1 + c2) / 2
    mean_c7 = mean_c.pow(7)
    g = 0.5 * (1 - torch.sqrt(mean_c7 / (mean_c7 + 25**7)))
    a1_prime = (1 + g) * a1
    a2_prime = (1 + g) * a2
    c1_prime = torch.sqrt(a1_prime.square() + b1.square() + 1e-12)
    c2_prime = torch.sqrt(a2_prime.square() + b2.square() + 1e-12)
    h1_prime = torch.rad2deg(torch.atan2(b1, a1_prime)).remainder(360)
    h2_prime = torch.rad2deg(torch.atan2(b2, a2_prime)).remainder(360)

    delta_l = l2 - l1
    delta_c = c2_prime - c1_prime
    delta_h_degrees = h2_prime - h1_prime
    delta_h_degrees = torch.where(
        delta_h_degrees > 180, delta_h_degrees - 360, delta_h_degrees
    )
    delta_h_degrees = torch.where(
        delta_h_degrees < -180, delta_h_degrees + 360, delta_h_degrees
    )
    delta_h_degrees = torch.where(
        (c1_prime * c2_prime) == 0, torch.zeros_like(delta_h_degrees), delta_h_degrees
    )
    delta_h = (
        2
        * torch.sqrt(c1_prime * c2_prime)
        * torch.sin(torch.deg2rad(delta_h_degrees) / 2)
    )

    mean_l = (l1 + l2) / 2
    mean_c_prime = (c1_prime + c2_prime) / 2
    hue_distance = (h1_prime - h2_prime).abs()
    mean_h = (h1_prime + h2_prime) / 2
    mean_h = torch.where(hue_distance > 180, mean_h + 180, mean_h).remainder(360)
    mean_h = torch.where((c1_prime * c2_prime) == 0, h1_prime + h2_prime, mean_h)
    t = (
        1
        - 0.17 * torch.cos(torch.deg2rad(mean_h - 30))
        + 0.24 * torch.cos(torch.deg2rad(2 * mean_h))
        + 0.32 * torch.cos(torch.deg2rad(3 * mean_h + 6))
        - 0.20 * torch.cos(torch.deg2rad(4 * mean_h - 63))
    )
    delta_theta = 30 * torch.exp(-((mean_h - 275) / 25).square())
    r_c = 2 * torch.sqrt(mean_c_prime.pow(7) / (mean_c_prime.pow(7) + 25**7))
    s_l = 1 + (0.015 * (mean_l - 50).square()) / torch.sqrt(20 + (mean_l - 50).square())
    s_c = 1 + 0.045 * mean_c_prime
    s_h = 1 + 0.015 * mean_c_prime * t
    r_t = -torch.sin(torch.deg2rad(2 * delta_theta)) * r_c
    lightness = delta_l / s_l
    chroma = delta_c / s_c
    hue = delta_h / s_h
    return torch.sqrt(
        (
            lightness.square() + chroma.square() + hue.square() + r_t * chroma * hue
        ).clamp_min(0)
        + 1e-12
    )


def _ssim_per_image(output: torch.Tensor, expected: torch.Tensor) -> torch.Tensor:
    """Measure local luminance, contrast, and structure similarity."""
    x, y = _luminance(output), _luminance(expected)
    kernel = min(11, x.shape[-2], x.shape[-1])
    if kernel % 2 == 0:
        kernel -= 1
    padding = kernel // 2
    mean_x = F.avg_pool2d(x, kernel, stride=1, padding=padding)
    mean_y = F.avg_pool2d(y, kernel, stride=1, padding=padding)
    variance_x = F.avg_pool2d(x.square(), kernel, 1, padding) - mean_x.square()
    variance_y = F.avg_pool2d(y.square(), kernel, 1, padding) - mean_y.square()
    covariance = F.avg_pool2d(x * y, kernel, 1, padding) - mean_x * mean_y
    c1, c2 = 0.01**2, 0.03**2
    score = ((2 * mean_x * mean_y + c1) * (2 * covariance + c2)) / (
        (mean_x.square() + mean_y.square() + c1) * (variance_x + variance_y + c2)
    )
    return _per_image_mean(score)


def metric_samples(
    output: torch.Tensor, expected: torch.Tensor
) -> dict[str, torch.Tensor]:
    """Calculate one value per photograph for each complementary quality axis."""
    output, expected = output.float(), expected.float()
    error = output - expected
    mse = _per_image_mean(error.square())
    lab_output, lab_expected = rgb_to_lab(output), rgb_to_lab(expected)
    output_x, output_y = image_gradients(output)
    expected_x, expected_y = image_gradients(expected)
    edge_error = (
        _per_image_mean((output_x - expected_x).abs())
        + _per_image_mean((output_y - expected_y).abs())
    ) / 2
    chroma_output = torch.sqrt(lab_output[:, 1].square() + lab_output[:, 2].square())
    chroma_expected = torch.sqrt(
        lab_expected[:, 1].square() + lab_expected[:, 2].square()
    )
    return {
        "mae": _per_image_mean(error.abs()),
        "psnrDb": -10 * torch.log10(mse.clamp_min(1e-10)),
        "ssim": _ssim_per_image(output, expected),
        "deltaE2000": _per_image_mean(ciede2000(lab_output, lab_expected)),
        "luminanceMae": _per_image_mean(
            (_luminance(output) - _luminance(expected)).abs()
        ),
        "chromaMae": _per_image_mean((chroma_output - chroma_expected).abs()),
        "edgeMae": edge_error,
        "clippingPercent": _per_image_mean(
            ((output <= 1 / 255) | (output >= 254 / 255))
            .any(dim=1, keepdim=True)
            .float()
        )
        * 100,
    }


def statistics_match(source: torch.Tensor, reference: torch.Tensor) -> torch.Tensor:
    """Apply the editor's bounded per-channel mean/spread reference baseline."""
    source_mean = source.mean((2, 3), keepdim=True)
    source_std = source.std((2, 3), correction=0, keepdim=True)
    reference_mean = reference.mean((2, 3), keepdim=True)
    reference_std = reference.std((2, 3), correction=0, keepdim=True)
    gain = (reference_std / source_std.clamp_min(0.015)).clamp(0.65, 1.5)
    return ((source - source_mean) * gain + reference_mean).clamp(0, 1)


def mismatch_references(
    reference: torch.Tensor, style_labels
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pair each item with a reference carrying a different style label.

    Rolling a batch can accidentally compare two references from the same FiveK
    expert. That is not a meaningful wrong-style control. This deterministic
    mapping searches forward for a different expert and returns the selected
    indices so the evaluation remains auditable.
    """
    count = reference.shape[0]
    indices = torch.arange(count, device=reference.device)
    if count < 2:
        return reference, indices
    if isinstance(style_labels, torch.Tensor):
        label_values = style_labels.detach().cpu().tolist()
    else:
        label_values = list(style_labels)
    if len(label_values) != count:
        raise ValueError("One style label is required for every reference")
    selected = []
    for index in range(count):
        candidate = next(
            (
                (index + offset) % count
                for offset in range(1, count)
                if label_values[(index + offset) % count] != label_values[index]
            ),
            index,
        )
        selected.append(candidate)
    indices = torch.tensor(selected, device=reference.device)
    return reference.index_select(0, indices), indices


def _bootstrap(values: np.ndarray, seed: int, samples: int = 1000) -> dict[str, float]:
    """Estimate a photograph-level 95% confidence interval for a mean."""
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        raise ValueError("Cannot summarize an empty metric")
    if len(values) == 1:
        lower = upper = float(values[0])
    else:
        generator = np.random.default_rng(seed)
        indices = generator.integers(0, len(values), size=(samples, len(values)))
        means = values[indices].mean(1)
        lower, upper = np.percentile(means, [2.5, 97.5])
    return {
        "mean": float(values.mean()),
        "ci95Lower": float(lower),
        "ci95Upper": float(upper),
        "sampleCount": int(len(values)),
    }


def _cluster_bootstrap(
    values: np.ndarray, scene_ids, seed: int, samples: int = 1000
) -> dict[str, float]:
    """Bootstrap scene means so repeated expert edits do not inflate certainty.

    FiveK contains five expert variants of each source scene. Averaging those
    variants within a scene and then resampling scenes gives each photograph one
    independent vote while retaining every expert measurement in its scene mean.
    """
    values = np.asarray(values, dtype=np.float64)
    scenes = np.asarray(scene_ids, dtype=object)
    if len(values) != len(scenes):
        raise ValueError("One source scene identifier is required per metric value")
    unique_scenes = tuple(dict.fromkeys(scenes.tolist()))
    scene_means = np.asarray(
        [values[scenes == scene].mean() for scene in unique_scenes], dtype=np.float64
    )
    result = _bootstrap(scene_means, seed, samples)
    result["sceneCount"] = result.pop("sampleCount")
    result["observationCount"] = int(len(values))
    return result


class MetricStore:
    """Collect per-image measurements while retaining expert identities."""

    def __init__(self) -> None:
        self.values: dict[str, list[float]] = defaultdict(list)
        self.experts: list[int] = []
        self.scenes: list[str] = []

    def add(
        self, metrics: dict[str, torch.Tensor], experts: torch.Tensor, scenes
    ) -> None:
        scene_values = list(scenes)
        batch_size = len(experts)
        if len(scene_values) != batch_size:
            raise ValueError("One source scene identifier is required per photograph")
        for name, values in metrics.items():
            if len(values) != batch_size:
                raise ValueError("Every metric must contain one value per photograph")
            self.values[name].extend(values.detach().cpu().tolist())
        self.experts.extend(experts.detach().cpu().tolist())
        self.scenes.extend(scene_values)

    def summary(self, seed: int = 42) -> dict:
        overall = {
            name: _cluster_bootstrap(np.asarray(values), self.scenes, seed + index)
            for index, (name, values) in enumerate(sorted(self.values.items()))
        }
        expert_array = np.asarray(self.experts)
        scene_array = np.asarray(self.scenes, dtype=object)
        by_expert = {}
        for expert_index, expert_name in enumerate(EXPERTS):
            selected = expert_array == expert_index
            if selected.any():
                by_expert[expert_name.upper()] = {
                    name: _cluster_bootstrap(
                        np.asarray(values)[selected],
                        scene_array[selected],
                        seed + expert_index,
                    )
                    for name, values in sorted(self.values.items())
                }
        return {"overall": overall, "byExpert": by_expert}


def paired_improvement(
    model: MetricStore, baseline: MetricStore, seed: int = 42
) -> dict[str, dict[str, float]]:
    """Report positive values when the model improves over a paired baseline."""
    higher_is_better = {"psnrDb", "ssim"}
    if model.scenes != baseline.scenes:
        raise ValueError("Paired comparisons require metrics in the same scene order")
    comparisons = {}
    for index, name in enumerate(sorted(model.values)):
        model_values = np.asarray(model.values[name])
        baseline_values = np.asarray(baseline.values[name])
        difference = (
            model_values - baseline_values
            if name in higher_is_better
            else baseline_values - model_values
        )
        comparisons[name] = _cluster_bootstrap(difference, model.scenes, seed + index)
    return comparisons


def coefficient_diagnostics(coefficients: torch.Tensor) -> dict[str, object]:
    """Detect a collapsed mixture that ignores most learned LUT bases."""
    coefficients = coefficients.float()
    entropy = -(coefficients * coefficients.clamp_min(1e-8).log()).sum(1)
    entropy /= math.log(coefficients.shape[1])
    dominant_counts = torch.bincount(
        coefficients.argmax(1), minlength=coefficients.shape[1]
    )
    return {
        "normalizedEntropyMean": float(entropy.mean()),
        "normalizedEntropyMinimum": float(entropy.min()),
        "meanBasisWeights": coefficients.mean(0).tolist(),
        "basisWeightStandardDeviations": coefficients.std(0, correction=0).tolist(),
        "dominantBasisCounts": dominant_counts.tolist(),
        "dominantBasisFraction": float(dominant_counts.max() / coefficients.shape[0]),
    }


def deployment_evidence_gates(
    identity_gain: dict,
    statistics_gain: dict,
    wrong_reference_gain: dict,
    reference_sensitivity_mean: float,
    coefficient_report: dict,
) -> dict[str, bool]:
    """Require statistical confidence, practical effect, and model diversity."""
    minimum = MINIMUM_PRACTICAL_EFFECTS
    return {
        "beatsIdentityDeltaE2000": (
            identity_gain["deltaE2000"]["ci95Lower"] > 0
            and identity_gain["deltaE2000"]["mean"] >= minimum["identityDeltaE2000"]
        ),
        "beatsIdentitySsim": (
            identity_gain["ssim"]["ci95Lower"] > 0
            and identity_gain["ssim"]["mean"] >= minimum["identitySsim"]
        ),
        "beatsStatisticsDeltaE2000": (
            statistics_gain["deltaE2000"]["ci95Lower"] > 0
            and statistics_gain["deltaE2000"]["mean"] >= minimum["statisticsDeltaE2000"]
        ),
        "correctReferenceBeatsShuffledDeltaE2000": (
            wrong_reference_gain["deltaE2000"]["ci95Lower"] > 0
            and wrong_reference_gain["deltaE2000"]["mean"]
            >= minimum["correctReferenceDeltaE2000"]
        ),
        "respondsToReference": (
            reference_sensitivity_mean >= minimum["referenceSensitivityMae"]
        ),
        "avoidsSingleBasisCollapse": (
            max(coefficient_report["meanBasisWeights"]) < 0.9
            and coefficient_report["dominantBasisFraction"] < 0.9
        ),
        "doesNotIncreaseClippingVsIdentity": (
            identity_gain["clippingPercent"]["ci95Lower"] >= 0
        ),
    }


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/data.py
"""Select and stage a small, scene-disjoint TST100K triplet experiment.

The upstream manifest is the pairing authority. Scene IDs, rather than rows,
determine the split: a photograph used as either content or reference cannot
appear in a different split. Images are downloaded only inside Kaggle scratch.
"""

from __future__ import annotations

import hashlib
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from threading import Lock
from urllib.parse import quote
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

import torch
import numpy as np
from PIL import Image, ImageOps
from torch.utils.data import Dataset

DATASET_REVISION = "63bf13833b75407c862d608c3d81281d0a35b61f"
BASE_URL = f"https://huggingface.co/datasets/ToneStyle/TST100K/resolve/{DATASET_REVISION}"
SPLITS = ("train", "validation", "test")


class RequestPacer:
    """Space image requests and share server backoff across worker threads."""

    def __init__(self, interval_seconds: float = 0.75):
        self.interval_seconds = interval_seconds
        self.next_request_at = 0.0
        self.lock = Lock()

    def wait(self) -> None:
        while True:
            with self.lock:
                remaining = self.next_request_at - time.monotonic()
                if remaining <= 0:
                    self.next_request_at = time.monotonic() + self.interval_seconds
                    return
            time.sleep(remaining)

    def pause_for(self, seconds: float) -> None:
        with self.lock:
            self.next_request_at = max(self.next_request_at, time.monotonic() + seconds)


def scene_id(path: str) -> str:
    """Map source and preset-rendered copies to one original photograph."""
    return Path(path).stem.split("_")[0]


def scene_split(identifier: str, seed: int) -> str:
    """Hash each photograph once; references cannot cross a split boundary."""
    digest = hashlib.sha256(f"{seed}:{identifier}".encode()).digest()
    bucket = int.from_bytes(digest[:8], "big") % 10
    return "train" if bucket < 8 else "validation" if bucket == 8 else "test"


def select_triplets(
    records: list[dict[str, str]], counts: tuple[int, int, int], seed: int
) -> dict[str, list[dict[str, str]]]:
    """Select a reproducible sample with distinct target scenes in each split.

    Triplets whose content and reference belong to different partitions are
    omitted. This costs data but prevents direct scene reuse across evaluation.
    """
    selected = {split: [] for split in SPLITS}
    used_content = {split: set() for split in SPLITS}
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{seed}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    for row in ranked:
        if set(row) != {"content", "reference", "gt"}:
            raise ValueError("Each TST100K record must have content, reference, and gt")
        content_id = scene_id(row["content"])
        reference_id = scene_id(row["reference"])
        if content_id == reference_id or scene_id(row["gt"]) != content_id:
            continue
        split = scene_split(content_id, seed)
        if scene_split(reference_id, seed) != split:
            continue
        if content_id in used_content[split] or len(selected[split]) >= counts[SPLITS.index(split)]:
            continue
        selected[split].append(row)
        used_content[split].add(content_id)
        if all(len(selected[name]) == counts[index] for index, name in enumerate(SPLITS)):
            break
    actual = tuple(len(selected[name]) for name in SPLITS)
    if actual != counts:
        raise ValueError(f"Insufficient disjoint triplets: requested {counts}, found {actual}")
    return selected


def fetch_manifest() -> list[dict[str, str]]:
    """Read the pinned upstream pairing manifest without storing it in Git."""
    with urlopen(f"{BASE_URL}/triplet.json", timeout=120) as response:
        return json.load(response)


def _download_one(root: Path, relative: str, max_bytes: int, pacer: RequestPacer) -> int:
    """Stream one image to Kaggle scratch, with bounded size and retries."""
    destination = root / relative
    if destination.is_file():
        return destination.stat().st_size
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"{BASE_URL}/{quote(relative, safe='/')}"
    for attempt in range(7):
        temporary = destination.with_name(destination.name + ".part")
        try:
            pacer.wait()
            with urlopen(url, timeout=90) as response:
                size = 0
                with temporary.open("wb") as output:
                    while chunk := response.read(1024 * 1024):
                        size += len(chunk)
                        if size > max_bytes:
                            raise ValueError(f"Image exceeds {max_bytes} bytes: {relative}")
                        output.write(chunk)
            with Image.open(temporary) as image:
                image.verify()
            temporary.replace(destination)
            return size
        except HTTPError as error:
            temporary.unlink(missing_ok=True)
            if error.code != 429 or attempt == 6:
                raise
            retry_after = error.headers.get("Retry-After")
            try:
                delay = max(1.0, float(retry_after)) if retry_after else 0.0
            except ValueError:
                delay = 0.0
            delay = min(300.0, max(delay, 20.0 * 2**attempt))
            pacer.pause_for(delay)
            print(f"Rate limited by data host; pausing image requests for {delay:.0f}s", flush=True)
        except (URLError, OSError, ValueError) as error:
            temporary.unlink(missing_ok=True)
            if attempt == 6 or isinstance(error, ValueError):
                raise
            time.sleep(min(30, 2**attempt))
    raise RuntimeError("Unreachable download retry state")


def stage_images(
    selected: dict[str, list[dict[str, str]]], root: Path, workers: int = 2
) -> dict[str, int]:
    """Download only selected files; the full 94 GB collection is never staged."""
    paths = sorted({path for rows in selected.values() for row in rows for path in row.values()})
    total_bytes = 0
    pacer = RequestPacer()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = {pool.submit(_download_one, root, path, 40_000_000, pacer): path for path in paths}
        for completed, task in enumerate(as_completed(pending), 1):
            total_bytes += task.result()
            if completed % 250 == 0 or completed == len(paths):
                print(json.dumps({"downloadedFiles": completed, "totalFiles": len(paths)}), flush=True)
    return {"files": len(paths), "bytes": total_bytes}


class TripletDataset(Dataset):
    """Decode aligned source/answer pixels and an independent style reference."""

    def __init__(self, rows: list[dict[str, str]], root: Path, size: int = 128):
        self.rows = rows
        self.root = root
        self.size = size

    def __len__(self) -> int:
        return len(self.rows)

    def _image(self, path: str) -> torch.Tensor:
        with Image.open(self.root / path) as loaded:
            image = ImageOps.exif_transpose(loaded).convert("RGB")
            image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
            pixels = torch.from_numpy(np.array(image, copy=True))
        return pixels.permute(2, 0, 1).float().div_(255)

    def __getitem__(self, index: int) -> dict:
        row = self.rows[index]
        return {
            "source": self._image(row["content"]),
            "reference": self._image(row["reference"]),
            "expected": self._image(row["gt"]),
            "scene": scene_id(row["content"]),
        }


def save_split_manifest(selected: dict, path: Path, seed: int) -> None:
    """Record every pairing and revision needed to reproduce the experiment."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps({"datasetRevision": DATASET_REVISION, "seed": seed, "splits": selected}, indent=2)
    )


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/train.py
"""Train a compact reference-conditioned LUT on TST100K triplets.

This pilot measures whether a correct reference improves a held-out target,
not merely whether generic retouching looks pleasant. Only validation selects
the checkpoint; the test set is evaluated once after training.
"""

from __future__ import annotations

import argparse
import json
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from tools.ml.reference_matcher.evaluation import metric_samples, statistics_match
from tools.ml.reference_matcher.losses import ReferenceMatchLoss, reference_ranking_loss
from tools.ml.reference_matcher.model import ReferenceLutModel, apply_lut
from .data import SPLITS, TripletDataset, fetch_manifest, save_split_manifest, select_triplets, stage_images


def mean_interval(values: list[float], seed: int) -> dict:
    """Report a 95% bootstrap interval across distinct target photographs."""
    samples = np.asarray(values, dtype=np.float64)
    if not len(samples):
        raise ValueError("Cannot report an empty evaluation set")
    generator = np.random.default_rng(seed)
    estimates = samples[generator.integers(0, len(samples), (1000, len(samples)))].mean(1)
    low, high = np.percentile(estimates, [2.5, 97.5])
    return {"mean": float(samples.mean()), "ci95": [float(low), float(high)], "n": len(samples)}


def image_batch(batch: dict, device: torch.device) -> tuple[torch.Tensor, ...]:
    """Move the three image tensors to GPU; keep metadata on the CPU."""
    return tuple(batch[key].to(device, non_blocking=True) for key in ("source", "reference", "expected"))


def autocast_context(device: torch.device):
    """Use a fast CUDA format while keeping local CPU checks in full precision."""
    if device.type != "cuda":
        return nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast("cuda", dtype=dtype)


@torch.no_grad()
def evaluate(model: ReferenceLutModel, loader: DataLoader, device: torch.device, seed: int) -> dict:
    """Measure correct, wrong, identity, and existing statistics matching."""
    model.eval()
    metrics = {
        name: {metric: [] for metric in ("deltaE2000", "mae", "ssim", "edgeMae", "clippingPercent")}
        for name in ("model", "wrongReference", "identity", "statistics")
    }
    difference = []
    started = time.perf_counter()
    for batch in loader:
        source, reference, expected = image_batch(batch, device)
        if len(source) < 2:
            raise ValueError("Evaluation needs at least two references per batch")
        wrong_reference = reference.roll(1, 0)
        with autocast_context(device):
            lut, _ = model(source, reference)
            wrong_lut, _ = model(source, wrong_reference)
            output = apply_lut(source, lut).float()
            wrong = apply_lut(source, wrong_lut).float()
        images = {
            "model": output,
            "wrongReference": wrong,
            "identity": source,
            "statistics": statistics_match(source, reference),
        }
        for name, image in images.items():
            samples = metric_samples(image, expected)
            for metric in metrics[name]:
                metrics[name][metric].extend(samples[metric].cpu().tolist())
        difference.extend((output - wrong).abs().flatten(1).mean(1).cpu().tolist())
    results = {
        name: {
            ("rgbMae" if metric == "mae" else metric): mean_interval(values, seed)
            for metric, values in sample_metrics.items()
        }
        for name, sample_metrics in metrics.items()
    }
    results["modelVsWrongReference"] = mean_interval(
        (np.asarray(metrics["wrongReference"]["deltaE2000"]) - np.asarray(metrics["model"]["deltaE2000"])).tolist(), seed
    )
    results["modelVsStatistics"] = mean_interval(
        (np.asarray(metrics["statistics"]["deltaE2000"]) - np.asarray(metrics["model"]["deltaE2000"])).tolist(), seed
    )
    results["modelVsIdentity"] = mean_interval(
        (np.asarray(metrics["identity"]["deltaE2000"]) - np.asarray(metrics["model"]["deltaE2000"])).tolist(), seed
    )
    results["modelVsStatisticsSsim"] = mean_interval(
        (np.asarray(metrics["model"]["ssim"]) - np.asarray(metrics["statistics"]["ssim"])).tolist(), seed
    )
    results["referenceSensitivityRgbMae"] = mean_interval(difference, seed)
    results["elapsedSeconds"] = time.perf_counter() - started
    return results


def edit_emphasis_loss(output: torch.Tensor, expected: torch.Tensor, source: torch.Tensor) -> torch.Tensor:
    """Give stronger expert edits more weight without changing inference.

    The batch-relative weights are bounded and renormalized. The channel-mean
    term asks the LUT to reach the intended global shift, not invent geometry.
    """
    edit_size = (expected - source).abs().flatten(1).mean(1).detach()
    weights = (0.75 + edit_size / edit_size.mean().clamp_min(1e-6)).clamp(0.75, 2.0)
    weights = weights / weights.mean()
    per_image = (output - expected).abs().flatten(1).mean(1)
    weighted_pixel = (weights * per_image).mean()
    channel_mean = (output.mean(dim=(2, 3)) - expected.mean(dim=(2, 3))).abs().mean()
    return 0.25 * weighted_pixel + 0.25 * channel_mean


def train_epoch(model, loader, optimizer, scaler, device, epoch: int, *, emphasize_edits: bool = False) -> dict:
    """Update the model using real triplets and explicit reference controls."""
    model.train()
    criterion = ReferenceMatchLoss()
    totals = {"loss": 0.0, "reconstruction": 0.0, "ranking": 0.0, "identity": 0.0}
    started = time.perf_counter()
    for step, batch in enumerate(loader):
        source, reference, expected = image_batch(batch, device)
        wrong_reference = reference.roll(1, 0)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            lut, _ = model(source, reference)
            wrong_lut, _ = model(source, wrong_reference)
            output = apply_lut(source, lut).float()
            wrong_output = apply_lut(source, wrong_lut).float()
            reconstruction, _ = criterion(output, expected, source, lut.float())
            ranking = reference_ranking_loss(output, wrong_output, expected, margin=0.01)
            # On every fourth step, teach a neutral reference to preserve the
            # source. This is a counterfactual training example, not a label.
            identity = source.new_zeros(())
            if step % 4 == 0:
                neutral_lut, _ = model(source, source)
                identity = (apply_lut(source, neutral_lut).float() - source).abs().mean()
            loss = reconstruction + 2 * ranking + 0.25 * identity
            if emphasize_edits:
                loss = loss + edit_emphasis_loss(output, expected, source)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        for key, value in (("loss", loss), ("reconstruction", reconstruction), ("ranking", ranking), ("identity", identity)):
            totals[key] += float(value.detach())
        if (step + 1) % 16 == 0:
            print(json.dumps({"epoch": epoch, "step": step + 1, "steps": len(loader), "loss": round(totals["loss"] / (step + 1), 5)}), flush=True)
    return {**{key: value / len(loader) for key, value in totals.items()}, "elapsedSeconds": time.perf_counter() - started}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/colorcheck-tst100k"))
    parser.add_argument("--scratch-dir", type=Path, default=Path("/kaggle/temp/colorcheck-tst100k"))
    parser.add_argument("--train-pairs", type=int, default=256)
    parser.add_argument("--validation-pairs", type=int, default=64)
    parser.add_argument("--test-pairs", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--size", type=int, default=128)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=2)
    args = parser.parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This controlled Kaggle pilot requires a CUDA GPU")
    if args.batch_size < 2 or any(value < 2 for value in (args.train_pairs, args.validation_pairs, args.test_pairs)):
        raise ValueError("Each split and batch must contain at least two references")

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    args.output_dir.mkdir(parents=True, exist_ok=True)
    records = fetch_manifest()
    selected = select_triplets(
        records, (args.train_pairs, args.validation_pairs, args.test_pairs), args.seed
    )
    save_split_manifest(selected, args.output_dir / "splits.json", args.seed)
    staged = stage_images(selected, args.scratch_dir)
    print(json.dumps({"staged": staged, "pairs": {key: len(value) for key, value in selected.items()}}), flush=True)

    device = torch.device("cuda")
    loaders = {
        split: DataLoader(
            TripletDataset(selected[split], args.scratch_dir, args.size),
            batch_size=args.batch_size,
            shuffle=split == "train",
            num_workers=args.workers,
            pin_memory=True,
            drop_last=split == "train",
        )
        for split in SPLITS
    }
    model = ReferenceLutModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=not torch.cuda.is_bf16_supported())
    history = []
    best_score = float("inf")
    for epoch in range(1, args.epochs + 1):
        training = train_epoch(model, loaders["train"], optimizer, scaler, device, epoch)
        validation = evaluate(model, loaders["validation"], device, args.seed)
        # Checkpoint choice rewards correct-reference quality and penalizes
        # a model that ignores which reference was supplied.
        score = validation["model"]["deltaE2000"]["mean"] - validation["modelVsWrongReference"]["mean"]
        history.append({"epoch": epoch, "train": training, "validation": validation, "selectionScore": score})
        (args.output_dir / "history.json").write_text(json.dumps(history, indent=2))
        print(json.dumps({"epoch": epoch, "validationDeltaE2000": validation["model"]["deltaE2000"]["mean"], "correctReferenceGain": validation["modelVsWrongReference"]["mean"], "score": score}), flush=True)
        if score < best_score:
            best_score = score
            torch.save({"epoch": epoch, "state_dict": model.state_dict(), "seed": args.seed}, args.output_dir / "best.pt")

    checkpoint = torch.load(args.output_dir / "best.pt", map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    test = evaluate(model, loaders["test"], device, args.seed + 1)
    gates = {
        "beatsIdentity": test["modelVsIdentity"]["ci95"][0] > 0,
        "beatsCurrentMatcher": test["modelVsStatistics"]["ci95"][0] > 0
        and test["modelVsStatistics"]["mean"] >= 0.5,
        "usesCorrectReference": test["modelVsWrongReference"]["ci95"][0] > 0
        and test["modelVsWrongReference"]["mean"] >= 0.25,
    }
    result = {
        "dataset": "ToneStyle/TST100K",
        "selectedCheckpointEpoch": checkpoint["epoch"],
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "test": test,
        "gates": gates,
        "allGatesPass": all(gates.values()),
        "note": "Research pilot only. No model is promoted to the site by this command.",
    }
    (args.output_dir / "summary.json").write_text(json.dumps(result, indent=2))
    print(json.dumps({"gates": gates, "test": {key: test[key]["mean"] for key in ("modelVsIdentity", "modelVsStatistics", "modelVsWrongReference")}}), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/prepare_v4.py
"""Freeze v4's final holdout and verify strong-edit validation coverage.

Run this on Kaggle CPU before training. It excludes every scene used by v3 or
its later audit, then checks that validation can test the stated hypothesis.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

from torch.utils.data import DataLoader

from .data import TripletDataset, fetch_manifest, scene_id, stage_images

FINAL_SEED = 6109
FINAL_COUNT = 256
STRONG_EDIT_THRESHOLD = 0.08


def mounted_file(kernel_slug: str, name: str) -> Path:
    """Resolve a mounted output without confusing two similarly named files."""
    matches = [
        path for path in Path("/kaggle/input").rglob(name)
        if kernel_slug in path.parts
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {name} from {kernel_slug}; found {matches}")
    return matches[0]


def select_final_rows(records: list[dict], old_splits: dict, audit_rows: list[dict]) -> list[dict]:
    """Select fixed, never-seen target and reference scenes by hash rank."""
    used = {
        scene_id(row[key])
        for rows in (*old_splits.values(), audit_rows)
        for row in rows
        for key in ("content", "reference")
    }
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{FINAL_SEED}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    selected = []
    targets = set()
    for row in ranked:
        target = scene_id(row["content"])
        reference = scene_id(row["reference"])
        if (
            target in used or reference in used or target == reference
            or target in targets or scene_id(row["gt"]) != target
        ):
            continue
        selected.append(row)
        targets.add(target)
        if len(selected) == FINAL_COUNT:
            break
    if len(selected) != FINAL_COUNT:
        raise ValueError(f"Expected {FINAL_COUNT} untouched targets; found {len(selected)}")
    return selected


def strong_edit_count(rows: list[dict], scratch: Path) -> int:
    """Count source/expert pairs with at least 0.08 mean RGB change."""
    loader = DataLoader(TripletDataset(rows, scratch, 128), batch_size=8, num_workers=2)
    return sum(
        int(((batch["expected"] - batch["source"]).abs().mean(dim=(1, 2, 3))
            >= STRONG_EDIT_THRESHOLD).sum())
        for batch in loader
    )


def main() -> None:
    output = Path("/kaggle/working/colorcheck-tst100k-v4-preflight")
    scratch = Path("/kaggle/temp/colorcheck-tst100k-v4-preflight")
    output.mkdir(parents=True, exist_ok=True)
    old = json.loads(mounted_file("colorcheck-tst100k-expanded-validation", "splits.json").read_text())
    audit = json.loads(mounted_file("colorcheck-tst100k-fresh-evaluation", "splits.json").read_text())
    final_rows = select_final_rows(fetch_manifest(), old["splits"], audit["rows"])
    payload = {"datasetRevision": old["datasetRevision"], "seed": FINAL_SEED, "rows": final_rows}
    manifest = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    (output / "final-holdout.json").write_text(json.dumps(payload, indent=2))
    checksum = hashlib.sha256(manifest.encode()).hexdigest()
    print(json.dumps({"finalTargets": len(final_rows), "manifestSha256": checksum}), flush=True)

    print(json.dumps({"stagedValidation": stage_images({"validation": old["splits"]["validation"]}, scratch)}), flush=True)
    strong = strong_edit_count(old["splits"]["validation"], scratch)
    result = {
        "finalTargets": len(final_rows),
        "finalSeed": FINAL_SEED,
        "manifestSha256": checksum,
        "validationTargets": len(old["splits"]["validation"]),
        "strongValidationTargets": strong,
        "strongEditThreshold": STRONG_EDIT_THRESHOLD,
        "ready": strong >= 20,
    }
    (output / "preflight.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result), flush=True)
    if not result["ready"]:
        raise RuntimeError("Fewer than 20 strong-edit validation targets; revise protocol before GPU training")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/prepare_selector.py
"""Reserve unseen TST100K scenes before fitting a deployment selector.

This preflight reads only JSON manifests. It never fetches photographs, opens
expert edits, or loads either candidate matcher.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

from .data import DATASET_REVISION, fetch_manifest, scene_id
from .prepare_v4 import mounted_file

SEED = 7201
COUNTS = {"train": 1024, "validation": 256, "final": 256}


def scene_set(rows: list[dict]) -> set[str]:
    """Count a photograph in either target or reference position."""
    return {scene_id(row[key]) for row in rows for key in ("content", "reference")}


def select_rows(records: list[dict], excluded: set[str], counts: dict[str, int] = COUNTS) -> dict:
    """Hash-rank eligible rows, keeping all three partitions scene-disjoint."""
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{SEED}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    partitions = {name: [] for name in counts}
    occupied = set(excluded)
    for name, count in counts.items():
        targets = set()
        for row in ranked:
            target = scene_id(row["content"])
            reference = scene_id(row["reference"])
            if (target in occupied or reference in occupied or target == reference
                    or scene_id(row["gt"]) != target or target in targets):
                continue
            partitions[name].append(row)
            targets.add(target)
            if len(partitions[name]) == count:
                break
        if len(partitions[name]) != count:
            raise ValueError(f"Insufficient unseen {name} targets: {len(partitions[name])}/{count}")
        occupied.update(scene_set(partitions[name]))
    return partitions


def checksum(payload: dict) -> str:
    """Hash canonical JSON, independent of whitespace in the saved file."""
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()


def main() -> None:
    output = Path("/kaggle/working/colorcheck-tst100k-selector-preflight")
    output.mkdir(parents=True, exist_ok=True)
    old = json.loads(mounted_file("colorcheck-tst100k-expanded-validation", "splits.json").read_text())
    audit = json.loads(mounted_file("colorcheck-tst100k-fresh-evaluation", "splits.json").read_text())
    v4 = json.loads(mounted_file("colorcheck-tst100k-v4-preflight", "final-holdout.json").read_text())
    if old["datasetRevision"] != DATASET_REVISION or v4["datasetRevision"] != DATASET_REVISION:
        raise RuntimeError("Pinned dataset revision changed")
    excluded = scene_set([
        row for rows in old["splits"].values() for row in rows
    ] + audit["rows"] + v4["rows"])
    partitions = select_rows(fetch_manifest(), excluded)
    payload = {"datasetRevision": DATASET_REVISION, "seed": SEED, "splits": partitions}
    (output / "selector-splits.json").write_text(json.dumps(payload, indent=2))
    result = {
        "ready": True,
        "counts": {name: len(rows) for name, rows in partitions.items()},
        "excludedScenes": len(excluded),
        "manifestSha256": checksum(payload),
        "finalSha256": checksum({"rows": partitions["final"]}),
        "note": "Manifest-only preflight; no photographs or expert edits opened.",
    }
    (output / "preflight.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/selector.py
"""Small, browser-portable signals for choosing a reference matcher.

Every feature is computed from images available when a user edits a photo.
The expert edit is deliberately absent from this module's interface.
"""

from __future__ import annotations

import torch

FEATURE_NAMES = (
    *(f"source_mean_{channel}" for channel in "rgb"),
    *(f"source_std_{channel}" for channel in "rgb"),
    *(f"reference_mean_{channel}" for channel in "rgb"),
    *(f"reference_std_{channel}" for channel in "rgb"),
    *(f"reference_minus_source_mean_{channel}" for channel in "rgb"),
    *(f"reference_minus_source_std_{channel}" for channel in "rgb"),
    *(f"lut_minus_source_mean_{channel}" for channel in "rgb"),
    *(f"statistics_minus_source_mean_{channel}" for channel in "rgb"),
    "lut_shift_mae", "statistics_shift_mae", "lut_clipping_fraction",
    "statistics_clipping_fraction", "candidate_disagreement_mae",
    "source_luminance_mean", "source_luminance_std",
    "reference_luminance_mean", "reference_luminance_std",
)


def _channel_mean(image: torch.Tensor) -> torch.Tensor:
    return image.mean(dim=(2, 3))


def _channel_std(image: torch.Tensor) -> torch.Tensor:
    return image.std(dim=(2, 3), correction=0)


def _luminance_summary(image: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    weights = image.new_tensor((0.2126, 0.7152, 0.0722))
    luminance = (image * weights[None, :, None, None]).sum(dim=1)
    return luminance.mean(dim=(1, 2)), luminance.std(dim=(1, 2), correction=0)


def _clipping(image: torch.Tensor) -> torch.Tensor:
    return ((image <= 1 / 255) | (image >= 254 / 255)).any(dim=1).float().mean(dim=(1, 2))


def selector_features(source: torch.Tensor, reference: torch.Tensor,
                      lut_output: torch.Tensor, statistics_output: torch.Tensor) -> torch.Tensor:
    """Return one fixed-schema feature row per photo, without ground truth."""
    if not all(image.shape == source.shape for image in (reference, lut_output, statistics_output)):
        raise ValueError("Source, reference, and candidate images must have the same shape")
    if source.ndim != 4 or source.shape[1] != 3:
        raise ValueError("Expected Bx3xHxW RGB tensors")
    source, reference = source.float(), reference.float()
    lut_output, statistics_output = lut_output.float(), statistics_output.float()
    source_luma, source_luma_std = _luminance_summary(source)
    reference_luma, reference_luma_std = _luminance_summary(reference)
    columns = (
        _channel_mean(source), _channel_std(source),
        _channel_mean(reference), _channel_std(reference),
        _channel_mean(reference) - _channel_mean(source),
        _channel_std(reference) - _channel_std(source),
        _channel_mean(lut_output - source),
        _channel_mean(statistics_output - source),
        (lut_output - source).abs().mean(dim=(1, 2, 3), keepdim=False)[:, None],
        (statistics_output - source).abs().mean(dim=(1, 2, 3))[:, None],
        _clipping(lut_output)[:, None], _clipping(statistics_output)[:, None],
        (lut_output - statistics_output).abs().mean(dim=(1, 2, 3))[:, None],
        source_luma[:, None], source_luma_std[:, None],
        reference_luma[:, None], reference_luma_std[:, None],
    )
    result = torch.cat(columns, dim=1)
    if result.shape[1] != len(FEATURE_NAMES) or not torch.isfinite(result).all():
        raise RuntimeError("Selector feature schema changed or produced nonfinite values")
    return result


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/train_selector.py
"""Train a small quality selector on fresh scenes without opening final data.

Frozen v3 and the existing statistics matcher make candidate images. The
expert edit labels which candidate is closer, but is never a model feature.
Only validation selects regularization and a conservative routing threshold.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from tools.ml.reference_matcher.evaluation import metric_samples, statistics_match
from tools.ml.reference_matcher.model import ReferenceLutModel, apply_lut
from .data import TripletDataset, stage_images
from .prepare_selector import checksum
from .prepare_v4 import mounted_file
from .selector import FEATURE_NAMES, selector_features
from .train import mean_interval

EXPECTED_SPLIT_SHA256 = "5cf65a621ca1c33475a1f3579b22caa4f0e3f33ec563df176202950e7272afa8"
EXPECTED_FINAL_SHA256 = "a0e8940a69c494bef7648949364d6fefe0a7187ef0ab575d41a0a54d280fe399"
SEED = 7201
TIE_BAND = 0.25
REGULARIZATION = (0.1, 1.0, 10.0)
THRESHOLDS = tuple(round(value / 100, 2) for value in range(50, 96, 5))


def labels_from_errors(lut_error: np.ndarray, statistics_error: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Training labels use expert error; ties are omitted, not forced."""
    advantage = statistics_error - lut_error
    return (advantage > TIE_BAND).astype(np.int64), np.abs(advantage) > TIE_BAND


def validation_metrics(
    lut_error: np.ndarray, stats_error: np.ndarray,
    wrong_lut_error: np.ndarray, wrong_stats_error: np.ndarray,
    lut_ssim: np.ndarray, stats_ssim: np.ndarray,
    lut_clipping: np.ndarray, stats_clipping: np.ndarray,
    select_lut: np.ndarray, wrong_select_lut: np.ndarray,
) -> dict:
    """Evaluate one fixed threshold with photo-level paired comparisons."""
    chosen_error = np.where(select_lut, lut_error, stats_error)
    wrong_error = np.where(wrong_select_lut, wrong_lut_error, wrong_stats_error)
    gain = stats_error - chosen_error
    chosen_ssim = np.where(select_lut, lut_ssim, stats_ssim)
    chosen_clipping = np.where(select_lut, lut_clipping, stats_clipping)
    interval = mean_interval(gain.tolist(), SEED)
    result = {
        "deltaEGainVsStatistics": interval,
        "nonregressionFraction": float(np.mean(gain >= -0.5)),
        "largeLossFraction": float(np.mean(gain < -2.0)),
        "correctReferenceGain": float(np.mean(wrong_error - chosen_error)),
        "lutSelectionFraction": float(np.mean(select_lut)),
        "meanSsim": float(np.mean(chosen_ssim)),
        "statisticsMeanSsim": float(np.mean(stats_ssim)),
        "meanClippingPercent": float(np.mean(chosen_clipping)),
        "statisticsMeanClippingPercent": float(np.mean(stats_clipping)),
        "meanDeltaE": float(np.mean(chosen_error)),
        "statisticsMeanDeltaE": float(np.mean(stats_error)),
    }
    result["gates"] = {
        "nonregression": result["nonregressionFraction"] >= 0.9,
        "practicalColorGain": interval["mean"] >= 0.5 and interval["ci95"][0] > 0,
        "correctReference": result["correctReferenceGain"] > 0,
        "selectsLutOftenEnough": result["lutSelectionFraction"] >= 0.2,
        "structure": result["meanSsim"] >= result["statisticsMeanSsim"] - 0.005,
        "clipping": result["meanClippingPercent"] <= result["statisticsMeanClippingPercent"] + 1.0,
    }
    result["allGatesPass"] = all(result["gates"].values())
    return result


@torch.inference_mode()
def extract_split(model: ReferenceLutModel, rows: list[dict], scratch: Path,
                  device: torch.device) -> dict[str, np.ndarray]:
    """Create features and evaluation columns without saving any photographs."""
    loader = DataLoader(TripletDataset(rows, scratch, 128), batch_size=8,
                        num_workers=2, pin_memory=device.type == "cuda")
    columns: dict[str, list[np.ndarray]] = {name: [] for name in (
        "features", "wrongFeatures", "lutError", "statsError", "wrongLutError",
        "wrongStatsError", "lutSsim", "statsSsim", "lutClipping", "statsClipping",
    )}
    model.eval()
    for batch in loader:
        source, reference, expected = (
            batch[key].to(device, non_blocking=True)
            for key in ("source", "reference", "expected")
        )
        if len(source) < 2:
            raise RuntimeError("Wrong-reference control needs batches of at least two")
        wrong_reference = reference.roll(1, 0)
        lut = apply_lut(source, model(source, reference)[0]).float()
        wrong_lut = apply_lut(source, model(source, wrong_reference)[0]).float()
        stats = statistics_match(source, reference)
        wrong_stats = statistics_match(source, wrong_reference)
        lut_metrics = metric_samples(lut, expected)
        stats_metrics = metric_samples(stats, expected)
        wrong_lut_metrics = metric_samples(wrong_lut, expected)
        wrong_stats_metrics = metric_samples(wrong_stats, expected)
        for name, value in (
            ("features", selector_features(source, reference, lut, stats)),
            ("wrongFeatures", selector_features(source, wrong_reference, wrong_lut, wrong_stats)),
            ("lutError", lut_metrics["deltaE2000"]),
            ("statsError", stats_metrics["deltaE2000"]),
            ("wrongLutError", wrong_lut_metrics["deltaE2000"]),
            ("wrongStatsError", wrong_stats_metrics["deltaE2000"]),
            ("lutSsim", lut_metrics["ssim"]),
            ("statsSsim", stats_metrics["ssim"]),
            ("lutClipping", lut_metrics["clippingPercent"]),
            ("statsClipping", stats_metrics["clippingPercent"]),
        ):
            columns[name].append(value.cpu().numpy())
    return {name: np.concatenate(values) for name, values in columns.items()}


def main() -> None:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import log_loss
    from sklearn.preprocessing import StandardScaler

    if not torch.cuda.is_available():
        raise RuntimeError("Enable Kaggle GPU for candidate generation")
    torch.set_num_threads(4)
    output = Path("/kaggle/working/colorcheck-tst100k-selector-training")
    scratch = Path("/kaggle/temp/colorcheck-tst100k-selector-training")
    output.mkdir(parents=True, exist_ok=True)
    preflight = json.loads(mounted_file("colorcheck-tst100k-selector-preflight", "preflight.json").read_text())
    splits = json.loads(mounted_file("colorcheck-tst100k-selector-preflight", "selector-splits.json").read_text())
    if (not preflight["ready"] or checksum(splits) != EXPECTED_SPLIT_SHA256
            or preflight["finalSha256"] != EXPECTED_FINAL_SHA256
            or checksum({"rows": splits["splits"]["final"]}) != EXPECTED_FINAL_SHA256):
        raise RuntimeError("Selector split preflight or sealed-final checksum changed")
    selected = {name: splits["splits"][name] for name in ("train", "validation")}
    if tuple(len(selected[name]) for name in selected) != (1024, 256):
        raise RuntimeError("Selector training and validation sizes changed")
    print(json.dumps({"staged": stage_images(selected, scratch)}), flush=True)

    device = torch.device("cuda")
    print(json.dumps({"gpu": torch.cuda.get_device_name(0)}), flush=True)
    checkpoint = torch.load(
        mounted_file("colorcheck-tst100k-expanded-validation", "best.pt"),
        map_location=device, weights_only=True,
    )
    if checkpoint["epoch"] != 8:
        raise RuntimeError("Expected the frozen v3 epoch-8 checkpoint")
    matcher = ReferenceLutModel().to(device).eval()
    matcher.load_state_dict(checkpoint["state_dict"])
    train = extract_split(matcher, selected["train"], scratch, device)
    validation = extract_split(matcher, selected["validation"], scratch, device)
    train_labels, train_keep = labels_from_errors(train["lutError"], train["statsError"])
    validation_labels, validation_keep = labels_from_errors(validation["lutError"], validation["statsError"])
    if len(set(train_labels[train_keep])) != 2 or len(set(validation_labels[validation_keep])) != 2:
        raise RuntimeError("Both selector classes must appear in training and validation")

    scaler = StandardScaler().fit(train["features"][train_keep])
    training_features = scaler.transform(train["features"][train_keep])
    validation_features = scaler.transform(validation["features"])
    candidates = []
    for strength in REGULARIZATION:
        candidate = LogisticRegression(C=strength, max_iter=1000, random_state=SEED)
        candidate.fit(training_features, train_labels[train_keep])
        probability = candidate.predict_proba(validation_features)[:, 1]
        score = log_loss(validation_labels[validation_keep], probability[validation_keep], labels=[0, 1])
        candidates.append((float(score), strength, candidate))
    loss, strength, selector = min(candidates, key=lambda row: (row[0], row[1]))
    correct_probability = selector.predict_proba(validation_features)[:, 1]
    wrong_probability = selector.predict_proba(scaler.transform(validation["wrongFeatures"]))[:, 1]
    reviewed = []
    for threshold in THRESHOLDS:
        measured = validation_metrics(
            validation["lutError"], validation["statsError"],
            validation["wrongLutError"], validation["wrongStatsError"],
            validation["lutSsim"], validation["statsSsim"],
            validation["lutClipping"], validation["statsClipping"],
            correct_probability >= threshold, wrong_probability >= threshold,
        )
        reviewed.append({"threshold": threshold, **measured})
    passing = [row for row in reviewed if row["allGatesPass"]]
    selected_threshold = max(
        passing, key=lambda row: (row["deltaEGainVsStatistics"]["mean"], row["threshold"])
    ) if passing else None
    result = {
        "dataset": "ToneStyle/TST100K", "splitSha256": EXPECTED_SPLIT_SHA256,
        "sealedFinalSha256": EXPECTED_FINAL_SHA256,
        "trainTargets": len(selected["train"]), "validationTargets": len(selected["validation"]),
        "trainingLabelsUsed": int(train_keep.sum()), "validationLabelsUsed": int(validation_keep.sum()),
        "chosenC": strength, "validationLogLoss": loss,
        "thresholds": reviewed,
        "selectedThreshold": selected_threshold["threshold"] if selected_threshold else None,
        "allGatesPass": selected_threshold is not None,
        "note": "Validation only; final-holdout photographs were not fetched or evaluated.",
    }
    (output / "summary.json").write_text(json.dumps(result, indent=2))
    (output / "selector-model.json").write_text(json.dumps({
        "featureNames": FEATURE_NAMES, "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(), "coefficients": selector.coef_[0].tolist(),
        "intercept": float(selector.intercept_[0]),
        "threshold": result["selectedThreshold"], "chosenC": strength,
        "splitSha256": EXPECTED_SPLIT_SHA256,
        "researchOnly": True,
    }, indent=2))
    print(json.dumps({"allGatesPass": result["allGatesPass"],
                      "selectedThreshold": result["selectedThreshold"],
                      "chosenC": strength}), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'tools.ml.tst100k.train_selector'], cwd='/kaggle/working', check=True)
